# 03 — Cypher básico: fazendo perguntas ao grafo

Você já tem um grafo carregado. Agora vamos aprender a **perguntar coisas a ele**.

Cypher é a linguagem de consulta do Neo4j. A boa notícia: se você já viu SQL, vai
reconhecer metade das ideias. E se nunca viu, melhor ainda — Cypher foi desenhado
para ser **desenhável**. A consulta se parece com o desenho do que você procura.

Este notebook é uma parada de fôlego entre a carga (notebook 02) e a investigação
de fraude (notebook 04). Nada aqui é sobre fraude: é só sobre aprender a
linguagem, usando os dados do banco que você acabou de montar.

> Pré-requisito: ter rodado o notebook `02_modelagem_e_carga.ipynb`.

In [ ]:
!pip install -q neo4j-rust-ext pandas python-dotenv

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda.
2. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o caminho para quem roda localmente.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

> ⚠️ **No Colab, cada notebook precisa de permissão para cada secret.** Ter criado
> o secret na sua conta não basta: abra o painel 🔑 e ative a chave
> **"Acesso ao notebook"** (*Notebook access*) para **este** notebook. Sem isso o
> secret é ignorado silenciosamente e o notebook volta a perguntar na tela.
>
> Se os secrets estiverem configurados (e liberados), a célula abaixo não pergunta
> nada — apenas conecta.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: Secrets do Colab > variável de ambiente (.env) > pergunta na tela."""
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except ImportError:
        pass  # não estamos no Colab
    except Exception as e:
        # O caso confuso: o secret existe, mas este notebook não tem permissão.
        # Sem este aviso, o notebook só voltaria a perguntar, sem explicar por quê.
        if "NotebookAccess" in type(e).__name__:
            print(f"⚠️  O secret '{nome}' existe, mas este notebook não tem acesso a ele.")
            print(f"    Abra o painel 🔑 e ative 'Acesso ao notebook' para '{nome}'.")

    if valor := os.environ.get(nome):
        return valor

    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

Uma função auxiliar para não repetir código: roda a query e devolve o resultado
como uma tabela do pandas, que o notebook exibe bonitinho.

In [ ]:
def cypher(query, **parametros):
    """Roda uma query Cypher e devolve o resultado como DataFrame."""
    registros, _, _ = driver.execute_query(query, database_=NEO4J_DATABASE, **parametros)
    return pd.DataFrame([r.data() for r in registros])

### Um CPF de exemplo — e por que usar parâmetros

Vários exercícios abaixo investigam **um** cliente específico. Como os CPFs são
gerados aleatoriamente no notebook 01, eles mudam a cada execução — então em vez de
escrever um CPF fixo na consulta, vamos escolher um do banco e passá-lo como
**parâmetro**.

Parâmetro em Cypher é `$nome`, e o valor vai separado da consulta. Faça isso sempre,
por dois motivos: evita injeção (o valor nunca é interpretado como código) e permite
ao Neo4j reaproveitar o plano de execução entre chamadas com valores diferentes.

Nunca monte consulta com f-string ou concatenação.

In [ ]:
# escolhemos um cliente movimentado, para os exemplos terem o que mostrar
resultado = cypher("""
    MATCH (c:Cliente)-[:REALIZOU]->(:Pix)-[:PARA]->(:Cliente)
    WITH c, count(*) AS pix_enviados
    MATCH (:Cliente)-[:REALIZOU]->(:Pix)-[:PARA]->(c)
    WITH c, pix_enviados, count(*) AS pix_recebidos
    RETURN c.cpf AS cpf, c.nome AS nome, pix_enviados, pix_recebidos
    ORDER BY pix_enviados + pix_recebidos DESC
    LIMIT 1
""")

CPF_EXEMPLO = resultado.iloc[0]["cpf"]
print(f"Vamos usar este cliente nos exemplos: {resultado.iloc[0]['nome']} (CPF {CPF_EXEMPLO})")
resultado

## 1. A anatomia de uma consulta

Toda consulta Cypher de leitura tem a mesma espinha dorsal:

```cypher
MATCH   (o padrão que eu procuro)
WHERE   (as condições que ele precisa satisfazer)
RETURN  (o que eu quero de volta)
```

Se você conhece SQL, o mapeamento é quase direto:

| SQL | Cypher | Diferença importante |
|---|---|---|
| `FROM tabela` | `MATCH (n:Label)` | em vez de tabela, um **padrão** |
| `WHERE ...` | `WHERE ...` | praticamente igual |
| `SELECT col` | `RETURN n.prop` | igual, mas vem por último |
| `JOIN ... ON` | `-[:RELACIONAMENTO]->` | **não existe JOIN**: você desenha a ligação |
| `LIMIT n` | `LIMIT n` | igual |

A diferença que muda tudo é a última linha da tabela. Em SQL você *reconstrói* a
ligação entre tabelas toda vez, comparando chaves. Em Cypher a ligação já está
gravada no dado — você só aponta para ela.

## 2. Encontrando nós

O padrão mais simples: um nó sozinho. Parênteses `()` desenham um nó, e `:Label`
diz de que tipo ele é.

Comece **sempre** com `LIMIT` quando estiver explorando — senão você traz o banco
inteiro para a tela.

In [ ]:
cypher("""
    MATCH (c:Cliente)
    RETURN c.cpf, c.nome, c.data_cadastro
    LIMIT 5
""")

Para achar **um** nó específico, coloque a propriedade entre chaves dentro do
próprio padrão. É o jeito idiomático em Cypher (e é o que usa o índice criado
pela constraint no notebook 02):

In [ ]:
cypher("""
    MATCH (c:Cliente {cpf: $cpf})
    RETURN c.cpf, c.nome
""", cpf=CPF_EXEMPLO)

O mesmo resultado, escrito com `WHERE`. As duas formas são equivalentes; use
chaves para igualdade simples e `WHERE` quando precisar de comparações,
intervalos ou lógica mais elaborada.

In [ ]:
cypher("""
    MATCH (t:Transacao)
    WHERE t.valor > 4900
    RETURN t.transacao_id, t.valor
    ORDER BY t.valor DESC
    LIMIT 5
""")

### Uma pegadinha: onde foi parar o "tipo"?

Instinto natural: escrever `RETURN t.tipo` para ver se é Pix, Boleto, Compra…
Experimente — vem `None`.

Por quê? Porque no notebook 02 nós decidimos que o tipo da transação seria uma
**label**, não uma propriedade. Cada transação foi criada como
`(:Transacao:Pix)`, e não como `(:Transacao {tipo: 'Pix'})`.

Para ler as labels de um nó, use a função `labels()`:

In [ ]:
cypher("""
    MATCH (t:Transacao)
    WHERE t.valor > 4900
    RETURN t.transacao_id, labels(t) AS labels, t.valor
    ORDER BY t.valor DESC
    LIMIT 5
""")

### Labels são filtros de graça

E aqui está a vantagem de ter feito assim: filtrar por tipo é só trocar o label —
sem `WHERE`, sem comparar strings, e usando o índice interno de labels:

In [ ]:
cypher("""
    MATCH (p:Pix)
    RETURN p.transacao_id, p.valor
    ORDER BY p.valor DESC
    LIMIT 5
""")

## 3. Seguindo relacionamentos — o pulo do gato

Aqui é onde Cypher deixa de parecer SQL. Para conectar dois nós, você **desenha a
seta**:

```
(a)-[:RELACIONAMENTO]->(b)
```

Leia da esquerda para a direita como uma frase: *"um Cliente REALIZOU uma
Transação"*.

In [ ]:
cypher("""
    MATCH (c:Cliente {cpf: $cpf})-[:REALIZOU]->(t:Transacao)
    RETURN c.nome, labels(t) AS tipo, t.valor
    LIMIT 10
""", cpf=CPF_EXEMPLO)

### Dois saltos

Nada impede continuar a frase. *"Um cliente realizou uma transação, que foi
para um empresa"* — três nós, duas setas, uma linha:

In [ ]:
cypher("""
    MATCH (c:Cliente {cpf: $cpf})-[:REALIZOU]->(t:Compra)-[:PARA]->(e:Empresa)
    RETURN c.nome, t.valor, e.nome AS empresa
    LIMIT 10
""", cpf=CPF_EXEMPLO)

**Pare um segundo aqui.** Essa consulta de uma linha, em SQL, seria um `JOIN`
de três tabelas com duas condições `ON`. E cada salto novo que você quisesse
adicionar seria mais um `JOIN`. Em Cypher, é mais uma seta na frase.

### A direção da seta importa

`-->` e `<--` significam coisas diferentes, e no nosso modelo isso separa "quem
mandou" de "quem recebeu" num Pix:

In [ ]:
# Pix que ESTE cliente enviou
enviados = cypher("""
    MATCH (c:Cliente {cpf: $cpf})-[:REALIZOU]->(:Pix)-[:PARA]->(destino:Cliente)
    RETURN destino.cpf AS para_quem, destino.nome AS nome
    LIMIT 5
""", cpf=CPF_EXEMPLO)

# Pix que ESTE cliente recebeu (note a seta invertida no fim)
recebidos = cypher("""
    MATCH (origem:Cliente)-[:REALIZOU]->(:Pix)-[:PARA]->(c:Cliente {cpf: $cpf})
    RETURN origem.cpf AS de_quem, origem.nome AS nome
    LIMIT 5
""", cpf=CPF_EXEMPLO)

print("Enviou Pix para:"); print(enviados)
print("\nRecebeu Pix de:"); print(recebidos)

Se a direção não importar para a sua pergunta, use um traço sem ponta: `-[:REL]-`.
Ele casa nos dois sentidos.

## 4. Contando e agrupando

`count()`, `sum()`, `avg()`, `max()` funcionam como em SQL — com uma diferença
que costuma agradar: **não existe `GROUP BY`**. Tudo que você coloca no `RETURN`
que não é uma função de agregação vira automaticamente chave de agrupamento.

In [ ]:
cypher("""
    MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)
    RETURN c.nome AS cliente,
           count(t) AS qtd_transacoes,
           round(sum(t.valor), 2) AS total_movimentado
    ORDER BY total_movimentado DESC
    LIMIT 10
""")

Aqui `c.nome` é a chave de agrupamento e `count(t)`/`sum(t.valor)` são as
agregações — o equivalente a um `GROUP BY c.nome` que você não precisou escrever.

E se quisermos agrupar **por tipo de transação**? Como o tipo é uma label,
precisamos primeiro transformar a lista de labels em linhas. `UNWIND` faz isso:
recebe uma lista e devolve uma linha para cada item dela.

In [ ]:
cypher("""
    MATCH (t:Transacao)
    UNWIND labels(t) AS tipo
    WITH tipo, t
    WHERE tipo <> 'Transacao'
    RETURN tipo,
           count(*) AS quantidade,
           round(avg(t.valor), 2) AS valor_medio,
           round(max(t.valor), 2) AS maior_valor
    ORDER BY quantidade DESC
""")

Duas coisas novas apareceram aí:

- **`UNWIND`** — transforma cada item de uma lista numa linha. Como cada transação
  tem `['Transacao', 'Pix']`, ela vira duas linhas; o `WHERE` descarta a genérica.
- **`WITH`** — passa resultados de uma etapa para a próxima. É o que permite
  filtrar (`WHERE`) algo que você acabou de calcular. Pense nele como um `RETURN`
  intermediário, que continua a consulta em vez de encerrá-la.

## 5. Quando um nó é compartilhado

Este é o padrão que vai importar muito no notebook 04, então vale entender agora,
sem a pressa da investigação.

Repare no formato do padrão abaixo: **duas setas apontando para o mesmo nó do
meio**. Ele encontra dois clientes diferentes que apontam para o mesmo
identificador — ou seja, que compartilham um RG, e-mail ou telefone.

O `WHERE c1.cpf < c2.cpf` evita que cada par apareça duas vezes
(uma como A-B e outra como B-A) e que um cliente case consigo mesmo.

In [ ]:
cypher("""
    MATCH (c1:Cliente)-[:TEM_RG]->(s:RG)<-[:TEM_RG]-(c2:Cliente)
    WHERE c1.cpf < c2.cpf
    RETURN c1.cpf AS cliente_1, c2.cpf AS cliente_2, s.valor AS rg_compartilhado
    LIMIT 10
""")

Se você quiser casar **qualquer** um dos três tipos de identificador de uma vez,
use `|` (leia como "ou"):

In [ ]:
cypher("""
    MATCH (c1:Cliente)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]->(id)<-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]-(c2:Cliente)
    WHERE c1.cpf < c2.cpf
    RETURN c1.cpf, c2.cpf, count(*) AS identificadores_em_comum
    ORDER BY identificadores_em_comum DESC
    LIMIT 10
""")

## 6. Caminhos de tamanho variável

Uma última ideia, e é a que SQL realmente não tem resposta boa: **e se eu não
souber quantos saltos preciso dar?**

`*1..3` significa "siga esse relacionamento de 1 a 3 vezes". Abaixo: clientes
alcançáveis a partir de um cliente por até 2 Pix encadeados — quem recebeu dele,
e quem recebeu de quem recebeu dele.

In [ ]:
cypher("""
    MATCH caminho = (c:Cliente {cpf: $cpf})
                    ((:Cliente)-[:REALIZOU]->(:Pix)-[:PARA]->(:Cliente)){1,2}
                    (alcancado:Cliente)
    RETURN alcancado.cpf AS alcancado, length(caminho) AS saltos
    LIMIT 10
""", cpf=CPF_EXEMPLO)

Em SQL, "de 1 a N saltos com N desconhecido" exige uma CTE recursiva — e cada
salto extra piora o desempenho de forma difícil de prever. Em Cypher é `{1,2}`.

Essa é, no fundo, a razão de existir de um banco de grafos: não é fazer o que SQL
não faz, é tornar **barato de perguntar** aquilo que em SQL é caro o bastante
para você desistir.

## 7. Exercícios

Tente escrever cada consulta antes de olhar a resposta. Rode e confira — errar a
sintaxe aqui é exatamente o objetivo do notebook.

1. Quantos clientes existem no banco?
2. Quais os 5 maiores valores de `Boleto`?
3. Quantas transações cada **empresa** recebeu?
4. Qual cliente enviou mais Pix (em quantidade)?
5. Quais clientes compartilham um **e-mail** com outro cliente?

In [ ]:
# Exercício 1 — Quantos clientes existem?
cypher("MATCH (c:Cliente) RETURN count(c) AS total_clientes")

In [ ]:
# Exercício 2 — Os 5 maiores boletos
cypher("""
    MATCH (b:Boleto)
    RETURN b.transacao_id, b.valor
    ORDER BY b.valor DESC
    LIMIT 5
""")

In [ ]:
# Exercício 3 — Transações por empresa
cypher("""
    MATCH (t:Transacao)-[:PARA]->(e:Empresa)
    RETURN e.nome AS empresa, count(t) AS qtd_transacoes
    ORDER BY qtd_transacoes DESC
    LIMIT 10
""")

In [ ]:
# Exercício 4 — Quem enviou mais Pix
cypher("""
    MATCH (c:Cliente)-[:REALIZOU]->(p:Pix)
    RETURN c.cpf, c.nome, count(p) AS qtd_pix
    ORDER BY qtd_pix DESC
    LIMIT 5
""")

In [ ]:
# Exercício 5 — Clientes que compartilham e-mail
cypher("""
    MATCH (c1:Cliente)-[:TEM_EMAIL]->(e:Email)<-[:TEM_EMAIL]-(c2:Cliente)
    WHERE c1.cpf < c2.cpf
    RETURN c1.cpf, c2.cpf, e.valor AS email_compartilhado
    LIMIT 10
""")

O resultado do exercício 5 deveria te deixar desconfiado. Dois clientes
diferentes, com o mesmo e-mail — e não é um caso isolado.

Guarde essa desconfiança: ela é o ponto de partida do próximo notebook.

## Colinha de Cypher

| Objetivo | Cypher |
|---|---|
| Achar nós de um tipo | `MATCH (c:Cliente) RETURN c LIMIT 10` |
| Achar um nó específico | `MATCH (c:Cliente {cpf: $cpf})` |
| Filtrar | `WHERE t.valor > 1000` |
| Seguir uma seta | `(a)-[:REALIZOU]->(b)` |
| Ignorar a direção | `(a)-[:REALIZOU]-(b)` |
| Vários tipos de seta | `(a)-[:TEM_RG\|TEM_EMAIL]->(b)` |
| De 1 a 3 saltos | `((a)-[:REL]->(b)){1,3}` |
| Contar | `RETURN count(*)` |
| Agrupar | é automático: tudo no `RETURN` que não agrega vira chave |
| Ordenar e limitar | `ORDER BY x DESC LIMIT 10` |
| Nó opcional (tipo LEFT JOIN) | `OPTIONAL MATCH (a)-[:REL]->(b)` |
| Criar/atualizar sem duplicar | `MERGE (c:Cliente {cpf: '...'})` |

Referência completa: [Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)
e a [folha de referência oficial](https://neo4j.com/docs/cypher-cheat-sheet/current/).

**Próximo passo:** `04_explorando_fraude.ipynb` — usar esse Cypher para investigar
de verdade.

In [ ]:
driver.close()